# Notebook 5: Consumer Narrative Preparation for NLP

## Objective

This notebook prepares CFPB consumer complaint narratives for downstream natural language processing.

The work includes:

- selecting complaints that contain narratives;
- validating narrative coverage across products and issue themes;
- creating NLP-ready text while preserving the original narrative;
- measuring narrative length;
- assessing repeated narratives;
- preparing modelling datasets for sentiment analysis and topic modelling.

The full taxonomy dataset is used for real complaint coverage and volumes. The balanced taxonomy dataset is used only when equal-product comparison is required.

No complaint records or taxonomy labels are redesigned in this notebook.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../outputs/notebook_5_nlp_preparation")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FULL_PATH = DATA_DIR / "cleaned_full_with_taxonomy.csv"
BALANCED_PATH = DATA_DIR / "balanced_data_with_taxonomy.csv"

In [4]:
full_df = pd.read_csv(
    FULL_PATH,
    low_memory=False
)

balanced_df = pd.read_csv(
    BALANCED_PATH,
    low_memory=False
)

print("Full taxonomy dataset:", full_df.shape)
print("Balanced taxonomy dataset:", balanced_df.shape)

Full taxonomy dataset: (566694, 29)
Balanced taxonomy dataset: (144697, 29)


In [5]:
required_columns = [
    "complaint_id",
    "product",
    "issue_theme",
    "issue",
    "issue_detail",
    "consumer_complaint_narrative"
]

missing_full = [
    column for column in required_columns
    if column not in full_df.columns
]

missing_balanced = [
    column for column in required_columns
    if column not in balanced_df.columns
]

assert not missing_full, f"Missing full-dataset columns: {missing_full}"
assert not missing_balanced, f"Missing balanced-dataset columns: {missing_balanced}"

print("All required NLP and taxonomy columns are available.")

All required NLP and taxonomy columns are available.


## 5.1 Narrative Availability

NLP methods require written consumer narratives. Complaints without narrative text remain in the master taxonomy datasets but cannot be used for text-based modelling.

Separate narrative-only working datasets are therefore created. A narrative is considered available when it is not missing and is not an empty string.

In [6]:
NARRATIVE_COLUMN = "consumer_complaint_narrative"


def has_usable_narrative(df):
    narrative_text = (
        df[NARRATIVE_COLUMN]
        .astype("string")
        .str.strip()
    )

    return narrative_text.notna() & narrative_text.ne("")


full_narrative_mask = has_usable_narrative(full_df)
balanced_narrative_mask = has_usable_narrative(balanced_df)

full_narratives = full_df.loc[
    full_narrative_mask
].copy()

balanced_narratives = balanced_df.loc[
    balanced_narrative_mask
].copy()

In [7]:
narrative_coverage_summary = pd.DataFrame({
    "dataset": [
        "Full dataset",
        "Balanced dataset"
    ],
    "total_complaints": [
        len(full_df),
        len(balanced_df)
    ],
    "with_narrative": [
        len(full_narratives),
        len(balanced_narratives)
    ]
})

narrative_coverage_summary["without_narrative"] = (
    narrative_coverage_summary["total_complaints"]
    - narrative_coverage_summary["with_narrative"]
)

narrative_coverage_summary["coverage_pct"] = (
    narrative_coverage_summary["with_narrative"]
    .div(narrative_coverage_summary["total_complaints"])
    .mul(100)
    .round(2)
)

display(narrative_coverage_summary)

,dataset,total_complaints,with_narrative,without_narrative,coverage_pct
0,Full dataset,566694,272466,294228,48.08
1,Balanced dataset,144697,79759,64938,55.12


## 5.2 Narrative coevrage by product

In [8]:

product_narrative_coverage = (
    full_df
    .assign(has_narrative=full_narrative_mask)
    .groupby("product")
    .agg(
        total_complaints=("has_narrative", "size"),
        with_narrative=("has_narrative", "sum")
    )
    .reset_index()
)

product_narrative_coverage["without_narrative"] = (
    product_narrative_coverage["total_complaints"]
    - product_narrative_coverage["with_narrative"]
)

product_narrative_coverage["coverage_pct"] = (
    product_narrative_coverage["with_narrative"]
    .div(product_narrative_coverage["total_complaints"])
    .mul(100)
    .round(2)
)

product_narrative_coverage = (
    product_narrative_coverage
    .sort_values("coverage_pct", ascending=False)
    .reset_index(drop=True)
)

display(product_narrative_coverage)

,product,total_complaints,with_narrative,without_narrative,coverage_pct
0,"Money transfer, virtual currency, or money service",84619,64862,19757,76.65
1,Checking or savings account,84194,51448,32746,61.11
2,Mortgage,24697,14029,10668,56.8
3,Credit card,89979,41517,48462,46.14
4,Debt collection,283205,100610,182595,35.53


### Narrative Coverage Findings

Narrative availability differs substantially across financial products.

Money-transfer complaints have the highest narrative coverage at 76.65%, followed by checking or savings accounts at 61.11% and mortgages at 56.80%. Credit cards have 46.14% coverage, while debt collection has the lowest coverage at 35.53%.

This means narrative-based results may overrepresent products with higher narrative availability and underrepresent debt collection complaints. Therefore, overall NLP results must be interpreted alongside product-level results.

Filtering the balanced dataset to narrative-bearing complaints also changes its original product balance. A separate equal-product narrative sample will therefore be created later for fair NLP comparisons.

## 5.3 Narrative Length Analysis

Narrative length is measured before NLP normalization. Character and word counts help identify unusually short or long narratives and guide later modelling decisions.

The original narrative column remains unchanged.

In [9]:
def add_narrative_length_features(df):
    result = df.copy()

    result["narrative_text"] = (
        result[NARRATIVE_COLUMN]
        .astype("string")
        .str.strip()
    )

    result["narrative_char_count"] = (
        result["narrative_text"].str.len()
    )

    result["narrative_word_count"] = (
        result["narrative_text"]
        .str.split()
        .str.len()
    )

    return result


full_narratives = add_narrative_length_features(
    full_narratives
)

balanced_narratives = add_narrative_length_features(
    balanced_narratives
)

In [10]:
length_summary = (
    full_narratives[
        ["narrative_char_count", "narrative_word_count"]
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .round(2)
)

display(length_summary)

,narrative_char_count,narrative_word_count
count,272466.0,272466.00
mean,1029.18,174.50
std,1149.04,194.66
min,39.0,4.00
1%,71.0,13.00
5%,150.0,27.00
25%,403.0,72.00
50%,678.0,107.00
75%,1265.0,218.00
95%,2876.75,489.00


In [11]:
product_length_summary = (
    full_narratives
    .groupby("product")
    .agg(
        narratives=("narrative_word_count", "size"),
        mean_words=("narrative_word_count", "mean"),
        median_words=("narrative_word_count", "median"),
        p95_words=(
            "narrative_word_count",
            lambda values: values.quantile(0.95)
        ),
        maximum_words=("narrative_word_count", "max")
    )
    .round(2)
    .sort_values("median_words", ascending=False)
    .reset_index()
)

display(product_length_summary)

,product,narratives,mean_words,median_words,p95_words,maximum_words
0,Mortgage,14029,298.19,218.0,798.0,5765
1,Credit card,41517,214.50,169.0,551.2,6469
2,Checking or savings account,51448,189.79,128.0,528.0,6218
3,Debt collection,100610,160.74,102.0,464.0,5220
4,"Money transfer, virtual currency, or money service",64862,131.35,102.0,316.0,5699


### Narrative Length Findings

The typical consumer narrative contains 107 words, while the average is approximately 175 words. Half of all narratives contain between 72 and 218 words.

Approximately 95% contain fewer than 489 words and 99% contain fewer than 897 words. A small number of narratives are considerably longer, with the maximum containing 6,469 words. These long narratives are not automatically errors and should not be deleted without inspection.

Narrative length also differs across products. Mortgage complaints are the most detailed, with a median of 218 words. Debt collection and money-transfer complaints have the shortest median length at 102 words.

These differences should be considered during NLP modelling, particularly when selecting maximum token lengths for transformer models.

## 5.4 Inspect narrative-length extremes

In [12]:
short_threshold = int(
    full_narratives["narrative_word_count"].quantile(0.01)
)

long_threshold = int(
    full_narratives["narrative_word_count"].quantile(0.99)
)

full_narratives["is_short_narrative"] = (
    full_narratives["narrative_word_count"]
    <= short_threshold
)

full_narratives["is_long_narrative"] = (
    full_narratives["narrative_word_count"]
    >= long_threshold
)

print("Short-narrative threshold:", short_threshold, "words")
print("Long-narrative threshold:", long_threshold, "words")

Short-narrative threshold: 13 words
Long-narrative threshold: 897 words


In [13]:
length_flag_summary = pd.DataFrame({
    "length_group": [
        "Short narratives",
        "Typical-length narratives",
        "Long narratives"
    ],
    "complaints": [
        full_narratives["is_short_narrative"].sum(),

        (
            ~full_narratives["is_short_narrative"]
            & ~full_narratives["is_long_narrative"]
        ).sum(),

        full_narratives["is_long_narrative"].sum()
    ]
})

length_flag_summary["share_pct"] = (
    length_flag_summary["complaints"]
    .div(len(full_narratives))
    .mul(100)
    .round(2)
)

display(length_flag_summary)

,length_group,complaints,share_pct
0,Short narratives,3219,1.18
1,Typical-length narratives,266516,97.82
2,Long narratives,2731,1.00


In [14]:
short_narrative_preview = (
    full_narratives
    .sort_values("narrative_word_count")
    .assign(
        narrative_preview=lambda df:
        df["narrative_text"].str.slice(0, 250)
    )
    [
        [
            "product",
            "issue_theme",
            "narrative_word_count",
            "narrative_preview"
        ]
    ]
    .head(15)
)

display(short_narrative_preview)

,product,issue_theme,narrative_word_count,narrative_preview
225902,"Money transfer, virtual currency, or money service","Fraud, scams and unauthorized activity",4,XX/XX/XXXX Cash App paymentXX/XX/XXXXXX/XX/XXXX
129426,Credit card,"Fees, interest and account funding",5,"unauthorized reporting, unverifiable ledger entry."
192642,"Money transfer, virtual currency, or money service","Transactions, payments and transfers",6,Company completed fraudulent transactions with funds.
518704,Debt collection,"Collection conduct, legal action and privacy",6,"Called very disrespectful, aggressive and threatening"
173580,Credit card,"Applications, marketing and disclosures",6,Received letter requesting more identity information
165481,Credit card,"Applications, marketing and disclosures",6,Fraud unauthorized transactions and opening accounts
490033,Debt collection,Debt validation and collection,6,INACCURATE INFORMATION FALSE INFORMATION BEING REPORTED
127395,Credit card,Financial hardship and repayment,6,American Express calling me aggressively everyday.
512385,Debt collection,Debt validation and collection,6,b'Collection \nMidland credit management inc \nXXXX\nXXXX
192738,"Money transfer, virtual currency, or money service","Transactions, payments and transfers",6,Unauthorized transactions from my checking account.


### Narrative-Length Decision

Approximately 97.82% of narratives fall within the typical-length range. The short narratives generally still communicate recognizable complaint information, such as fraud, threatening communication or inaccurate reporting.

Short and long narratives will therefore be retained. The length flags will be preserved for later quality analysis and transformer-token management rather than being used to delete records.

The preview also identified a residual byte-string or line-break artifact. A corpus-wide formatting audit is conducted before text normalization.

## 5.5 Residual text aftefact audits

In [15]:


text_series = (
    full_narratives["narrative_text"]
    .astype("string")
)

artifact_checks = {
    "byte_string_prefix": text_series.str.match(
        r"""^\s*b['"]""",
        na=False
    ),

    "actual_line_break": text_series.str.contains(
        r"[\r\n]",
        regex=True,
        na=False
    ),

    "literal_slash_n_or_r": text_series.str.contains(
        r"\\[nr]",
        regex=True,
        na=False
    ),

    "html_entity": text_series.str.contains(
        r"&(?:amp|lt|gt|quot|apos|#\d+);",
        regex=True,
        case=False,
        na=False
    ),

    "redaction_placeholder": text_series.str.contains(
        r"\bX{2,}\b",
        regex=True,
        na=False
    )
}

artifact_audit = pd.DataFrame({
    "artifact": artifact_checks.keys(),
    "complaints": [
        mask.sum()
        for mask in artifact_checks.values()
    ]
})

artifact_audit["share_pct"] = (
    artifact_audit["complaints"]
    .div(len(full_narratives))
    .mul(100)
    .round(2)
)

display(artifact_audit)

,artifact,complaints,share_pct
0,byte_string_prefix,1018,0.37
1,actual_line_break,116112,42.62
2,literal_slash_n_or_r,1067,0.39
3,html_entity,0,0.00
4,redaction_placeholder,169991,62.39


In [16]:
#Inspect byte prefix records
byte_prefix_preview = (
    full_narratives.loc[
        artifact_checks["byte_string_prefix"],
        [
            "complaint_id",
            "product",
            "narrative_text"
        ]
    ]
    .assign(
        narrative_preview=lambda df:
        df["narrative_text"].str.slice(0, 300)
    )
    .drop(columns="narrative_text")
    .head(15)
)

display(byte_prefix_preview)

,complaint_id,product,narrative_preview
30779,13028565,Checking or savings account,"b'Subject: Formal Request for Refund of XXXX in Repeated and Improper ACH Return Charges\n\nTo VyStar Credit Union,\n\nMy name is XXXXXXXX XXXX XX..."
30792,13025360,Checking or savings account,b'My name is XXXX XXXX please add this new issue I need to be complaint to cfbp case number XXXX-XXXX My name is XXXX XXXX I need to dispute pa...
30919,13047306,Checking or savings account,"b""I have had cash app for years I can't tell the exact date I've opened it but I had done my taxes and I had my taxes directly deposited into my c..."
30997,13031797,Checking or savings account,b'I responded to a text on my phone ostensibly from XXXX XXXX a respected organization back on XXXX XXXX. I did not think to check the URL. It tu...
31369,13100368,Checking or savings account,"b'Solicited by written offer of a bonus of XXXX by Wells Fargo both online and by direct mail, on XXXX XXXX, I meet with XXXX XXXX XXXX Bank Repre..."
31825,13158689,Checking or savings account,"b""CHIME XXXX XXXX XXXX XXXXXXXX XXXXXXXX CA XXXX \nsupport@chime.com,\nsupport@chimebank.com,\nchime@accounts.chime.com,\nXXXX \nhttps://www.chime..."
32488,13233556,Checking or savings account,"b'On XXXX XXXX I had a Hotel charge at the XXXXXXXX XXXX XXXX from my cash app checking account, included in the charge was a XXXX deposit, which ..."
32490,13233137,Checking or savings account,"b'I am filing a complaint against Bank Of America due to inadequate customer service and unfair practices, which violate the Consumer Financial Pr..."
32726,13293288,Checking or savings account,b'Capital one closed my account sir so they said they did in XXXX XXXX No activity until a deposit from irs landed XXXX The bank failed to noti...
32811,13040111,Checking or savings account,"b""On XXXX XXXX, Citibank unepectedly blocked my checking account without providing a reason. The very next day, led to be deposited into that acco..."


## 5.6 Residual Formatting Correction

A small proportion of narratives remain stored as Python byte-string representations. These wrappers are decoded to recover the underlying consumer text.

Actual and escaped line breaks are converted into spaces so that each narrative forms one continuous text document. Repeated whitespace is also standardized.

CFPB privacy-redaction placeholders are retained during this step and will be handled separately during model-text preparation.

The original `consumer_complaint_narrative` column remains unchanged.

In [17]:
import ast


def decode_byte_literal(text):
    """Decode a Python byte-string representation when present."""

    if pd.isna(text):
        return text

    text = str(text)

    if not re.match(r"""^\s*b['"]""", text):
        return text

    try:
        parsed_value = ast.literal_eval(text)

        if isinstance(parsed_value, bytes):
            return parsed_value.decode(
                "utf-8",
                errors="replace"
            )

    except (ValueError, SyntaxError):
        return text

    return text


def normalize_narrative_formatting(series):
    """Decode byte wrappers and standardize whitespace."""

    result = series.astype("string").copy()

    byte_mask = result.str.match(
        r"""^\s*b['"]""",
        na=False
    )

    result.loc[byte_mask] = (
        result.loc[byte_mask]
        .apply(decode_byte_literal)
    )

    # Replace literal escaped line-break characters
    result = result.str.replace(
        r"\\[nrt]",
        " ",
        regex=True
    )

    # Replace actual line breaks and tabs
    result = result.str.replace(
        r"[\r\n\t]+",
        " ",
        regex=True
    )

    # Standardize repeated whitespace
    result = result.str.replace(
        r"\s+",
        " ",
        regex=True
    ).str.strip()

    return result

In [18]:
full_narratives["narrative_text"] = (
    normalize_narrative_formatting(
        full_narratives["narrative_text"]
    )
)

balanced_narratives["narrative_text"] = (
    normalize_narrative_formatting(
        balanced_narratives["narrative_text"]
    )
)

print("Narrative formatting correction completed.")

Narrative formatting correction completed.


In [19]:
formatted_text = (
    full_narratives["narrative_text"]
    .astype("string")
)

post_formatting_audit = pd.DataFrame({
    "artifact": [
        "byte_string_prefix",
        "actual_line_break",
        "literal_slash_n_or_r"
    ],
    "remaining_complaints": [
        formatted_text.str.match(
            r"""^\s*b['"]""",
            na=False
        ).sum(),

        formatted_text.str.contains(
            r"[\r\n]",
            regex=True,
            na=False
        ).sum(),

        formatted_text.str.contains(
            r"\\[nr]",
            regex=True,
            na=False
        ).sum()
    ]
})

display(post_formatting_audit)

,artifact,remaining_complaints
0,byte_string_prefix,330
1,actual_line_break,0
2,literal_slash_n_or_r,0


### Residual Byte-Wrapper Handling

Strict byte-string decoding successfully corrected most affected narratives. A further 330 records contained irregular quotation or escape patterns that prevented automatic decoding.

For these remaining records, only the outer byte-string wrapper is removed. This targeted fallback does not alter unaffected narratives or delete complaint text.

In [20]:
def strip_residual_byte_wrapper(text):
    """Remove only the outer b'...' or b"..." wrapper."""

    if pd.isna(text):
        return text

    text = str(text).strip()

    wrapper_match = re.match(
        r"""^b(['"])""",
        text
    )

    if not wrapper_match:
        return text

    quote_character = wrapper_match.group(1)

    # Remove leading b' or b"
    text = text[2:]

    # Remove matching final quote when present
    if text.endswith(quote_character):
        text = text[:-1]

    # Correct commonly escaped quotation marks
    text = (
        text
        .replace(r"\'", "'")
        .replace(r'\"', '"')
    )

    return text.strip()

In [21]:
for narrative_df in [
    full_narratives,
    balanced_narratives
]:
    remaining_byte_mask = (
        narrative_df["narrative_text"]
        .astype("string")
        .str.match(
            r"""^\s*b['"]""",
            na=False
        )
    )

    narrative_df.loc[
        remaining_byte_mask,
        "narrative_text"
    ] = (
        narrative_df.loc[
            remaining_byte_mask,
            "narrative_text"
        ]
        .apply(strip_residual_byte_wrapper)
    )

    # Standardize whitespace again after fallback
    narrative_df["narrative_text"] = (
        narrative_df["narrative_text"]
        .astype("string")
        .str.replace(
            r"\\[nrt]",
            " ",
            regex=True
        )
        .str.replace(
            r"[\r\n\t]+",
            " ",
            regex=True
        )
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
        .str.strip()
    )

In [22]:
formatted_text = (
    full_narratives["narrative_text"]
    .astype("string")
)

final_formatting_audit = pd.DataFrame({
    "artifact": [
        "byte_string_prefix",
        "actual_line_break",
        "literal_slash_n_or_r"
    ],
    "remaining_complaints": [
        formatted_text.str.match(
            r"""^\s*b['"]""",
            na=False
        ).sum(),

        formatted_text.str.contains(
            r"[\r\n]",
            regex=True,
            na=False
        ).sum(),

        formatted_text.str.contains(
            r"\\[nr]",
            regex=True,
            na=False
        ).sum()
    ]
})

display(final_formatting_audit)

assert final_formatting_audit[
    "remaining_complaints"
].eq(0).all()

print("All residual formatting artifacts were corrected.")

,artifact,remaining_complaints
0,byte_string_prefix,0
1,actual_line_break,0
2,literal_slash_n_or_r,0


All residual formatting artifacts were corrected.


## 5.7 Model-Ready Narrative Text

Different NLP tasks require different levels of text preparation.

The `narrative_text` field preserves readable language, punctuation and capitalization. It is suitable for transformer-based sentiment analysis.

A second field, `narrative_topic_text`, is created for classical NLP and topic modelling. In this version:

- text is converted to lowercase;
- URLs and email addresses are removed;
- privacy-redaction placeholders are removed;
- standalone numbers and punctuation are removed;
- repeated whitespace is standardized.

Stop-word removal, lemmatization and token truncation are not performed here because these decisions depend on the downstream model.

In [23]:
def create_topic_text(series):
    text = series.astype("string").copy()

    # Keep visible text from Markdown-style links
    text = text.str.replace(
        r"\[([^\]]+)\]\((?:https?://|www\.)[^)]+\)",
        r"\1",
        regex=True
    )

    # Remove remaining URLs
    text = text.str.replace(
        r"(?:https?://|www\.)\S+",
        " ",
        regex=True
    )

    # Remove email addresses
    text = text.str.replace(
        r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b",
        " ",
        regex=True
    )

    # Remove CFPB redaction placeholders
    text = text.str.replace(
        r"\b[Xx]{2,}\b",
        " ",
        regex=True
    )

    # Convert to lowercase
    text = text.str.lower()

    # Remove standalone numbers
    text = text.str.replace(
        r"\b\d+(?:[.,]\d+)?\b",
        " ",
        regex=True
    )

    # Retain letters, spaces and apostrophes
    text = text.str.replace(
        r"[^a-z\s']",
        " ",
        regex=True
    )

    # Standardize whitespace
    text = text.str.replace(
        r"\s+",
        " ",
        regex=True
    ).str.strip()

    return text

In [24]:
full_narratives["narrative_topic_text"] = (
    create_topic_text(
        full_narratives["narrative_text"]
    )
)

balanced_narratives["narrative_topic_text"] = (
    create_topic_text(
        balanced_narratives["narrative_text"]
    )
)

full_narratives["topic_word_count"] = (
    full_narratives["narrative_topic_text"]
    .str.split()
    .str.len()
)

balanced_narratives["topic_word_count"] = (
    balanced_narratives["narrative_topic_text"]
    .str.split()
    .str.len()
)

In [25]:
model_text_validation = pd.DataFrame({
    "dataset": [
        "Full narratives",
        "Balanced narratives"
    ],
    "rows": [
        len(full_narratives),
        len(balanced_narratives)
    ],
    "missing_topic_text": [
        full_narratives["narrative_topic_text"].isna().sum(),
        balanced_narratives["narrative_topic_text"].isna().sum()
    ],
    "empty_topic_text": [
        full_narratives["narrative_topic_text"].eq("").sum(),
        balanced_narratives["narrative_topic_text"].eq("").sum()
    ],
    "median_topic_words": [
        full_narratives["topic_word_count"].median(),
        balanced_narratives["topic_word_count"].median()
    ]
})

display(model_text_validation)

,dataset,rows,missing_topic_text,empty_topic_text,median_topic_words
0,Full narratives,272466,0,12,106.0
1,Balanced narratives,79759,0,2,110.0


## 5.8 Low-Information Narrative Assessment

A small number of narratives become empty after removing privacy placeholders, numbers and punctuation. These records are retained in the narrative dataset because their original text remains available.

However, empty or extremely short normalized text may not contain enough meaningful vocabulary for topic modelling. Their frequency and content are therefore examined before defining topic-model eligibility.

In [26]:
topic_length_groups = pd.cut(
    full_narratives["topic_word_count"],
    bins=[-1, 0, 2, 4, 9, float("inf")],
    labels=[
        "0 words",
        "1–2 words",
        "3–4 words",
        "5–9 words",
        "10+ words"
    ]
)

topic_length_summary = (
    topic_length_groups
    .value_counts(sort=False)
    .rename_axis("topic_text_length")
    .reset_index(name="complaints")
)

topic_length_summary["share_pct"] = (
    topic_length_summary["complaints"]
    .div(len(full_narratives))
    .mul(100)
    .round(3)
)

display(topic_length_summary)

,topic_text_length,complaints,share_pct
0,0 words,12,0.004
1,1–2 words,4,0.001
2,3–4 words,21,0.008
3,5–9 words,1028,0.377
4,10+ words,271401,99.609


In [27]:
low_information_preview = (
    full_narratives.loc[
        full_narratives["topic_word_count"] < 5,
        [
            "complaint_id",
            "product",
            "issue_theme",
            "narrative_text",
            "narrative_topic_text",
            "topic_word_count"
        ]
    ]
    .sort_values("topic_word_count")
    .assign(
        original_preview=lambda df:
        df["narrative_text"].str.slice(0, 250)
    )
    .drop(columns="narrative_text")
    .head(25)
)

display(low_information_preview)

,complaint_id,product,issue_theme,narrative_topic_text,topic_word_count,original_preview
555679,16105815,Debt collection,Debt validation and collection,,0,XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX
441338,18232817,Debt collection,Debt validation and collection,,0,XXXX XXXX XXXX XXXXXXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX {$630.00}
212105,11595183,"Money transfer, virtual currency, or money service","Transactions, payments and transfers",,0,XXXX. {$500.00} {$650.00}.. {$1400.00} XXXX. {$96.00} {$430.00} {$550.00} {$820.00} XXXX XXXX XXXX {$400.00} XXXX. {$720.00} XXXX XXXX XXXX XXXX X...
201499,11641490,"Money transfer, virtual currency, or money service","Transactions, payments and transfers",,0,XXXX XXXX {$100.00} XXXX XXXX {$100.00} XXXX XXXX {$100.00} XXXX. XXXX {$100.00}
512555,14711891,Debt collection,Debt validation and collection,,0,XXXXXXXX XXXX {$3000.00} XXXXXXXX XXXX {$1200.00} XXXX XXXX {$13000.00}
74903,17619346,Checking or savings account,"Account management, access and closure",,0,XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX X...
99783,12276105,Credit card,Credit reporting and identity protection,,0,XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX X...
125658,13978559,Credit card,"Transactions, payments and transfers",,0,XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX
489481,14471437,Debt collection,Debt validation and collection,,0,XXXX {$15.00} XXXX {$12.00} XXXX {$5.00} XXXX {$10.00} XXXX {$12.00}
240488,14383560,"Money transfer, virtual currency, or money service","Fraud, scams and unauthorized activity",,0,"XX/XX/2025 XXXX, XXXX, XXXX, XXXX - XXXX, XXXX ( XXXX ) XXXX,XXXX XXXX XXXX"


### Topic-Modelling Eligibility Decision

Narratives containing fewer than five normalized words provide insufficient vocabulary for reliable unsupervised topic identification.

These complaints are not deleted. They remain available for complaint-level analysis and sentiment processing, but they are excluded from the topic-modelling corpus through an eligibility flag.

A narrative is considered topic-model eligible when its normalized text contains at least five words.

## 5.9 Create topic-model eligibility flag

In [28]:
MINIMUM_TOPIC_WORDS = 5

full_narratives["topic_model_eligible"] = (
    full_narratives["topic_word_count"]
    >= MINIMUM_TOPIC_WORDS
)

balanced_narratives["topic_model_eligible"] = (
    balanced_narratives["topic_word_count"]
    >= MINIMUM_TOPIC_WORDS
)

In [29]:
topic_eligibility_summary = pd.DataFrame({
    "dataset": [
        "Full narratives",
        "Balanced narratives"
    ],
    "total_narratives": [
        len(full_narratives),
        len(balanced_narratives)
    ],
    "eligible_for_topic_model": [
        full_narratives["topic_model_eligible"].sum(),
        balanced_narratives["topic_model_eligible"].sum()
    ]
})

topic_eligibility_summary["excluded_from_topic_model"] = (
    topic_eligibility_summary["total_narratives"]
    - topic_eligibility_summary["eligible_for_topic_model"]
)

topic_eligibility_summary["eligible_pct"] = (
    topic_eligibility_summary["eligible_for_topic_model"]
    .div(topic_eligibility_summary["total_narratives"])
    .mul(100)
    .round(3)
)

display(topic_eligibility_summary)

,dataset,total_narratives,eligible_for_topic_model,excluded_from_topic_model,eligible_pct
0,Full narratives,272466,272429,37,99.986
1,Balanced narratives,79759,79751,8,99.990


In [30]:
full_topic_corpus = (
    full_narratives.loc[
        full_narratives["topic_model_eligible"]
    ]
    .copy()
)

balanced_topic_corpus = (
    balanced_narratives.loc[
        balanced_narratives["topic_model_eligible"]
    ]
    .copy()
)

print("Full topic corpus:", full_topic_corpus.shape)
print("Balanced topic corpus:", balanced_topic_corpus.shape)

Full topic corpus: (272429, 36)
Balanced topic corpus: (79751, 34)


## 5.10 Align narrative dataset fields


In [31]:
# Apply the same full-dataset length thresholds
# to the balanced narrative dataset
balanced_narratives["is_short_narrative"] = (
    balanced_narratives["narrative_word_count"]
    <= short_threshold
)

balanced_narratives["is_long_narrative"] = (
    balanced_narratives["narrative_word_count"]
    >= long_threshold
)

# Confirm that both datasets contain the same fields
missing_from_balanced = (
    set(full_narratives.columns)
    - set(balanced_narratives.columns)
)

missing_from_full = (
    set(balanced_narratives.columns)
    - set(full_narratives.columns)
)

print("Missing from balanced:", missing_from_balanced)
print("Missing from full:", missing_from_full)

assert not missing_from_balanced
assert not missing_from_full

# Keep columns in the same order
balanced_narratives = balanced_narratives[
    full_narratives.columns
]

# Recreate the eligible corpora
full_topic_corpus = full_narratives.loc[
    full_narratives["topic_model_eligible"]
].copy()

balanced_topic_corpus = balanced_narratives.loc[
    balanced_narratives["topic_model_eligible"]
].copy()

print("Full topic corpus:", full_topic_corpus.shape)
print("Balanced topic corpus:", balanced_topic_corpus.shape)

Missing from balanced: set()
Missing from full: set()
Full topic corpus: (272429, 36)
Balanced topic corpus: (79751, 36)


## 5.11 Repeated Narrative Assessment

Some consumers, companies or complaint-support services may submit identical or template-based narratives multiple times.

Repeated narratives remain valid complaint records and must be retained for volume analysis. However, using every repeated copy while fitting a topic model could cause frequently copied templates to dominate the discovered topics.

Repetition is therefore measured using the normalized topic text. A unique-text corpus will later be used for topic discovery, while the complete narrative dataset will remain available for complaint-frequency analysis.

In [32]:
topic_text_frequency = (
    full_topic_corpus["narrative_topic_text"]
    .value_counts()
)

repeated_text_groups = (
    topic_text_frequency > 1
)

rows_in_repeated_groups = (
    full_topic_corpus["narrative_topic_text"]
    .duplicated(keep=False)
    .sum()
)

duplicate_rows_beyond_first = (
    len(full_topic_corpus)
    - full_topic_corpus["narrative_topic_text"].nunique()
)

duplicate_summary = pd.DataFrame({
    "metric": [
        "Eligible topic documents",
        "Unique normalized narratives",
        "Repeated unique texts",
        "Rows belonging to repeated groups",
        "Duplicate rows beyond first copy",
        "Largest repetition group"
    ],
    "value": [
        len(full_topic_corpus),
        full_topic_corpus["narrative_topic_text"].nunique(),
        repeated_text_groups.sum(),
        rows_in_repeated_groups,
        duplicate_rows_beyond_first,
        topic_text_frequency.max()
    ]
})

display(duplicate_summary)

,metric,value
0,Eligible topic documents,272429
1,Unique normalized narratives,210678
2,Repeated unique texts,5505
3,Rows belonging to repeated groups,67256
4,Duplicate rows beyond first copy,61751
5,Largest repetition group,12721


In [33]:
repeated_text_preview = (
    topic_text_frequency
    .head(10)
    .rename_axis("narrative_topic_text")
    .reset_index(name="frequency")
)

repeated_text_preview["text_preview"] = (
    repeated_text_preview["narrative_topic_text"]
    .str.slice(0, 250)
)

display(
    repeated_text_preview[
        ["frequency", "text_preview"]
    ]
)

,frequency,text_preview
0,12721,i am filing a complaint against cash app block inc due to inadequate customer service and unfair practices which violate the consumer financial pr...
1,5610,there are collection accounts on my report that i believe contain inaccurate information under my rights pursuant to usc e b and usc i i am entitl...
2,5131,i am writing to formally express my concerns regarding zelle 's handling of disputes particularly in light of the recent cfpb lawsuit highlighting...
3,3201,i understand the importance of removing any incorrect information from my credit report according to fcra b could you please review the attached d...
4,3009,i want to stress that i did not give written permission for these specific transactions to be included in my consumer report at no point did i aut...
5,2697,i am writing to file a complaint against navy federal credit union regarding multiple overdraft fees i have incurred despite having sufficient fun...
6,1976,could you please take a moment to review and examine the attached documents i believe your expertise will be invaluable in ensuring their accuracy...
7,1473,please blocked and removed i respectfully request the elimination of these fraudulent accounts from my credit report as i never authorized their i...
8,1445,i am filing a complaint against cash app block inc due to inadequate customer service and unfair practices which violate the consumer financial pr...
9,916,i am to formally express my concerns regarding zelles handling of disputes particularly in light of the recent cfpb lawsuit highlighting significa...


### Repeated-Narrative Modelling Decision

Approximately one-quarter of the eligible topic documents belong to repeated-text groups. The largest normalized narrative template appears 12,721 times.

Repeated complaint records remain important for measuring complaint volume. However, allowing thousands of identical templates into topic-model training would cause template wording to dominate the discovered topics.

Two corpora are therefore maintained:

1. **Full topic corpus** – retains every eligible complaint for topic assignment and prevalence analysis.
2. **Unique topic-training corpus** – contains one copy of each normalized narrative and is used to discover topics.

After training, topic labels can be merged back onto every complaint using the normalized narrative text.

 ## 5.12 Create unique topic-training corpus

In [34]:
full_narratives["normalized_text_frequency"] = (
    full_narratives["narrative_topic_text"]
    .map(topic_text_frequency)
    .fillna(0)
    .astype(int)
)

balanced_narratives["normalized_text_frequency"] = (
    balanced_narratives["narrative_topic_text"]
    .map(topic_text_frequency)
    .fillna(0)
    .astype(int)
)

full_narratives["is_repeated_narrative"] = (
    full_narratives["normalized_text_frequency"] > 1
)

balanced_narratives["is_repeated_narrative"] = (
    balanced_narratives["normalized_text_frequency"] > 1
)

/var/folders/kf/zl0ktfsd2d577bprgzdycbzr0000gn/T/ipykernel_17113/362879225.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  balanced_narratives["normalized_text_frequency"] = (
/var/folders/kf/zl0ktfsd2d577bprgzdycbzr0000gn/T/ipykernel_17113/362879225.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  balanced_narratives["is_repeated_narrative"] = (


In [35]:
unique_topic_corpus = (
    topic_text_frequency
    .rename_axis("narrative_topic_text")
    .reset_index(name="narrative_frequency")
)

unique_topic_corpus.insert(
    0,
    "topic_document_id",
    range(1, len(unique_topic_corpus) + 1)
)

display(unique_topic_corpus.head())

,topic_document_id,narrative_topic_text,narrative_frequency
0,1,i am filing a complaint against cash app block inc due to inadequate customer service and unfair practices which violate the consumer financial pr...,12721
1,2,there are collection accounts on my report that i believe contain inaccurate information under my rights pursuant to usc e b and usc i i am entitl...,5610
2,3,i am writing to formally express my concerns regarding zelle 's handling of disputes particularly in light of the recent cfpb lawsuit highlighting...,5131
3,4,i understand the importance of removing any incorrect information from my credit report according to fcra b could you please review the attached d...,3201
4,5,i want to stress that i did not give written permission for these specific transactions to be included in my consumer report at no point did i aut...,3009


In [36]:
assert len(unique_topic_corpus) == 210678

assert (
    unique_topic_corpus["narrative_topic_text"]
    .duplicated()
    .sum()
    == 0
)

assert (
    unique_topic_corpus["narrative_frequency"].sum()
    == len(full_topic_corpus)
)

print("Unique topic-training corpus:", unique_topic_corpus.shape)
print("Duplicate normalized texts: 0")
print(
    "Represented complaint records:",
    unique_topic_corpus["narrative_frequency"].sum()
)

Unique topic-training corpus: (210678, 3)
Duplicate normalized texts: 0
Represented complaint records: 272429


## 5.13 Equal-Product Narrative Corpus

The original balanced dataset contained approximately equal product samples. However, narrative availability differs by product, so filtering it to narrative-bearing complaints made the resulting NLP corpus unequal.

For fair product-level NLP comparisons, the same number of eligible narratives is sampled from each product. The sample size is determined by the product with the fewest eligible narratives.

This equal-product corpus is used only for comparative analysis. It does not represent real complaint volume.

In [37]:
# Recreate the corpora so they include all newly added metadata
full_topic_corpus = (
    full_narratives.loc[
        full_narratives["topic_model_eligible"]
    ]
    .copy()
)

balanced_topic_corpus = (
    balanced_narratives.loc[
        balanced_narratives["topic_model_eligible"]
    ]
    .copy()
)

In [38]:
eligible_narratives_by_product = (
    balanced_topic_corpus["product"]
    .value_counts()
    .rename_axis("product")
    .reset_index(name="eligible_narratives")
)

display(eligible_narratives_by_product)

comparison_sample_size = (
    eligible_narratives_by_product[
        "eligible_narratives"
    ]
    .min()
)

print(
    "Narratives sampled per product:",
    comparison_sample_size
)

,product,eligible_narratives
0,"Money transfer, virtual currency, or money service",22973
1,Checking or savings account,18299
2,Mortgage,14029
3,Credit card,13855
4,Debt collection,10595


Narratives sampled per product: 10595


In [39]:
equal_product_parts = []

for product_name, product_group in (
    balanced_topic_corpus.groupby("product")
):
    product_sample = product_group.sample(
        n=comparison_sample_size,
        random_state=42
    )

    equal_product_parts.append(product_sample)

equal_product_topic_corpus = (
    pd.concat(
        equal_product_parts,
        ignore_index=True
    )
)

equal_product_counts = (
    equal_product_topic_corpus["product"]
    .value_counts()
    .rename_axis("product")
    .reset_index(name="narratives")
)

display(equal_product_counts)

assert equal_product_counts["narratives"].nunique() == 1
assert (
    equal_product_topic_corpus["complaint_id"]
    .duplicated()
    .sum()
    == 0
)

print(
    "Equal-product topic corpus:",
    equal_product_topic_corpus.shape
)

,product,narratives
0,Checking or savings account,10595
1,Credit card,10595
2,Debt collection,10595
3,"Money transfer, virtual currency, or money service",10595
4,Mortgage,10595


Equal-product topic corpus: (52975, 37)


## 5.14 Final NLP-preparation validation

In [40]:
for narrative_df in [
    full_narratives,
    balanced_narratives
]:
    narrative_df["narrative_char_count"] = (
        narrative_df["narrative_text"]
        .str.len()
    )

    narrative_df["narrative_word_count"] = (
        narrative_df["narrative_text"]
        .str.split()
        .str.len()
    )

    narrative_df["is_short_narrative"] = (
        narrative_df["narrative_word_count"]
        <= short_threshold
    )

    narrative_df["is_long_narrative"] = (
        narrative_df["narrative_word_count"]
        >= long_threshold
    )

In [41]:
full_topic_corpus = (
    full_narratives.loc[
        full_narratives["topic_model_eligible"]
    ]
    .copy()
)

balanced_topic_corpus = (
    balanced_narratives.loc[
        balanced_narratives["topic_model_eligible"]
    ]
    .copy()
)

In [42]:
final_nlp_validation = pd.DataFrame({
    "check": [
        "Full narrative records",
        "Balanced narrative records",
        "Full topic-eligible records",
        "Unique topic-training texts",
        "Equal-product comparison records",
        "Missing readable text",
        "Missing normalized topic text",
        "Duplicate complaint IDs in full narratives",
        "Duplicate texts in unique training corpus",
        "Products in equal comparison corpus"
    ],
    "value": [
        len(full_narratives),
        len(balanced_narratives),
        len(full_topic_corpus),
        len(unique_topic_corpus),
        len(equal_product_topic_corpus),
        full_narratives["narrative_text"].isna().sum(),
        full_narratives["narrative_topic_text"].isna().sum(),
        full_narratives["complaint_id"].duplicated().sum(),
        unique_topic_corpus[
            "narrative_topic_text"
        ].duplicated().sum(),
        equal_product_topic_corpus["product"].nunique()
    ]
})

display(final_nlp_validation)

assert len(full_narratives) == 272466
assert len(balanced_narratives) == 79759
assert len(full_topic_corpus) == 272429
assert len(unique_topic_corpus) == 210678
assert len(equal_product_topic_corpus) == 52975

assert full_narratives["narrative_text"].notna().all()
assert full_narratives["narrative_topic_text"].notna().all()
assert full_narratives["complaint_id"].is_unique
assert unique_topic_corpus["narrative_topic_text"].is_unique
assert equal_product_topic_corpus["product"].nunique() == 5

print("All Notebook 5 NLP-preparation checks passed.")

,check,value
0,Full narrative records,272466
1,Balanced narrative records,79759
2,Full topic-eligible records,272429
3,Unique topic-training texts,210678
4,Equal-product comparison records,52975
5,Missing readable text,0
6,Missing normalized topic text,0
7,Duplicate complaint IDs in full narratives,0
8,Duplicate texts in unique training corpus,0
9,Products in equal comparison corpus,5


All Notebook 5 NLP-preparation checks passed.


## 5.15 Saving NLP-Prepared Outputs

The prepared narrative data is saved for downstream sentiment analysis and topic modelling.

The full and balanced narrative datasets retain complaint-level metadata and all preparation flags. Separate corpora are saved for unique-text topic training and equal-product comparison.

In [43]:
# Prepared complaint-level narrative datasets
full_narratives_path = (
    DATA_DIR / "full_narratives_prepared.csv"
)

balanced_narratives_path = (
    DATA_DIR / "balanced_narratives_prepared.csv"
)

# Specialized modelling corpora
unique_topic_path = (
    DATA_DIR / "unique_topic_training_corpus.csv"
)

equal_product_path = (
    DATA_DIR / "equal_product_nlp_corpus.csv"
)

full_narratives.to_csv(
    full_narratives_path,
    index=False
)

balanced_narratives.to_csv(
    balanced_narratives_path,
    index=False
)

unique_topic_corpus.to_csv(
    unique_topic_path,
    index=False
)

equal_product_topic_corpus.to_csv(
    equal_product_path,
    index=False
)

In [44]:
narrative_coverage_summary.to_csv(
    OUTPUT_DIR / "narrative_coverage_summary.csv",
    index=False
)

product_narrative_coverage.to_csv(
    OUTPUT_DIR / "product_narrative_coverage.csv",
    index=False
)

length_summary.reset_index().rename(
    columns={"index": "statistic"}
).to_csv(
    OUTPUT_DIR / "narrative_length_summary.csv",
    index=False
)

product_length_summary.to_csv(
    OUTPUT_DIR / "product_length_summary.csv",
    index=False
)

artifact_audit.to_csv(
    OUTPUT_DIR / "artifact_audit_before.csv",
    index=False
)

final_formatting_audit.to_csv(
    OUTPUT_DIR / "artifact_audit_after.csv",
    index=False
)

topic_eligibility_summary.to_csv(
    OUTPUT_DIR / "topic_eligibility_summary.csv",
    index=False
)

duplicate_summary.to_csv(
    OUTPUT_DIR / "duplicate_narrative_summary.csv",
    index=False
)

equal_product_counts.to_csv(
    OUTPUT_DIR / "equal_product_narrative_counts.csv",
    index=False
)

final_nlp_validation.to_csv(
    OUTPUT_DIR / "final_nlp_validation.csv",
    index=False
)

print("All Notebook 5 outputs saved successfully.")

All Notebook 5 outputs saved successfully.


In [45]:
saved_dataset_paths = [
    full_narratives_path,
    balanced_narratives_path,
    unique_topic_path,
    equal_product_path
]

saved_dataset_check = pd.DataFrame({
    "file": [
        path.name
        for path in saved_dataset_paths
    ],
    "exists": [
        path.exists()
        for path in saved_dataset_paths
    ],
    "size_mb": [
        round(
            path.stat().st_size / (1024 ** 2),
            2
        )
        if path.exists()
        else None
        for path in saved_dataset_paths
    ]
})

display(saved_dataset_check)

assert saved_dataset_check["exists"].all()

print("All prepared NLP datasets verified.")

,file,exists,size_mb
0,full_narratives_prepared.csv,True,1164.27
1,balanced_narratives_prepared.csv,True,373.05
2,unique_topic_training_corpus.csv,True,213.93
3,equal_product_nlp_corpus.csv,True,254.00


All prepared NLP datasets verified.


# Notebook 5 Completion Summary

Notebook 5 prepared the CFPB consumer narratives for sentiment analysis and topic modelling.

## Work completed

- Selected complaints containing usable consumer narratives.
- Preserved the original taxonomy-enriched complaint datasets.
- Measured narrative availability overall and by product.
- Calculated narrative character and word lengths.
- Retained short and long narratives while adding quality flags.
- Audited residual byte-string, line-break and HTML artifacts.
- Corrected 1,018 byte-string narratives without deleting complaint records.
- Created readable formatted narrative text.
- Created normalized topic-modelling text.
- Preserved readable text separately for transformer sentiment analysis.
- Identified records with insufficient topic-modelling vocabulary.
- Created a topic-model eligibility flag.
- Assessed exact repetition after text normalization.
- Created a unique-text corpus for unbiased topic discovery.
- Created an equal-product corpus for fair comparative NLP analysis.
- Validated all final corpora and saved the prepared outputs.

## Principal findings

The full dataset contained 272,466 complaints with narratives, representing 48.08% of all complaints.

Narrative availability differed substantially across products:

- Money transfer: 76.65%
- Checking or savings: 61.11%
- Mortgage: 56.80%
- Credit card: 46.14%
- Debt collection: 35.53%

The median narrative contained 107 words. Mortgage narratives were generally the longest, while debt collection and money-transfer narratives were shorter.

Only 37 full-dataset narratives contained fewer than five normalized words, leaving 272,429 eligible documents for topic analysis.

Repeated text was an important modelling concern. The eligible corpus contained 210,678 unique normalized narratives, while 67,256 complaint records belonged to repeated-text groups. The largest template appeared 12,721 times.

A unique-text training corpus was therefore created for topic discovery, while all complaint records were retained for topic prevalence and volume analysis.

The equal-product NLP corpus contains 10,595 narratives for each of the five products, producing 52,975 comparison records.

## Final outputs

- `full_narratives_prepared.csv`
- `balanced_narratives_prepared.csv`
- `unique_topic_training_corpus.csv`
- `equal_product_nlp_corpus.csv`

The prepared narratives are now ready for sentiment analysis and topic modelling.